# 00 — Environment check

Verifies the `oversquash` conda env is wired correctly **before** spending time on experiments:
PyTorch + PyTorch Geometric import, CPU/GPU, and the local editable `aiq-quivers` (with the 2026-06 fixes).

If any cell fails, fix the env (see `README.md`) before proceeding to `01_neighborsmatch.ipynb`.

In [ ]:
import sys, platform
print('Python', sys.version)
print('Platform', platform.platform())

In [ ]:
import torch, torch_geometric
print('torch       ', torch.__version__)
print('torch_geom  ', torch_geometric.__version__)
print('CUDA avail  ', torch.cuda.is_available())
print('device      ', 'cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# aiq-quivers must be importable (pip install -e ../quivers_analysis).
import aiq
from aiq.path_algebra import PathAlgebra, Ideal, QuotientAlgebra
from aiq.gnn import over_squashing_diagnostic
print('aiq-quivers', aiq.__version__, 'at', aiq.__file__)

# Sanity: is_algebraic_morphism was a stub before 2026-06; confirm the fix is present.
from aiq.morphisms import AIQMorphism
print('AIQMorphism.is_algebraic_morphism present:', hasattr(AIQMorphism, 'is_algebraic_morphism'))

In [ ]:
# This package.
import oversquash
from oversquash import layers, models, ideal_bridge, transforms, diagnostic, data, train
print('oversquash', oversquash.__version__)
print('OK — all imports resolved. Proceed to 01_neighborsmatch.ipynb')

In [ ]:
# Tiny end-to-end smoke test of the bridge on a 4-node diamond quiver
# (the ACT_en.tex Example ex:efecto_ideal graph): two parallel length-2 paths
# 1->2->4 and 1->3->4 should collapse from raw_mult=2 to effective_mult=1.
import numpy as np
edge_index = np.array([[0, 0, 1, 2], [1, 2, 3, 3]])  # 0->1,0->2,1->3,2->3
plan = ideal_bridge.build_quotient_plan(edge_index, num_nodes=4, max_length=2)
print('raw   mult (0->3, g=2):', plan.raw_mult.get((0, 3, 2)))
print('eff   mult (0->3, g=2):', plan.effective_mult.get((0, 3, 2)))
print('compression ratio     :', round(plan.compression_ratio(), 3))
assert plan.raw_mult.get((0, 3, 2)) == 2, 'expected 2 parallel walks'
assert plan.effective_mult.get((0, 3, 2)) == 1, 'kQ/I should collapse them to 1'
print('\nBridge smoke test PASSED — kQ/I collapses the two parallel paths.')

In [ ]:
# Core mechanism check: the multi-hop walk operators on a NeighborsMatch tree.
# The effective (kQ/I) operator must satisfy eff <= raw elementwise and strictly
# reduce total walk multiplicity (otherwise the quotient layer cannot help).
import torch
from oversquash.data import make_neighborsmatch_tree
from oversquash.ideal_bridge import walk_operators

g = torch.Generator().manual_seed(0)
tree = make_neighborsmatch_tree(radius=3, generator=g)
ei, N = tree.edge_index.numpy(), tree.x.size(0)
raw, eff = walk_operators(ei, N, max_length=4)
tot_raw = sum(R.sum() for R in raw); tot_eff = sum(E.sum() for E in eff)
for gi in range(4):
    assert (eff[gi] <= raw[gi] + 1e-9).all(), f'eff>raw at g={gi+1}'
    print(f'g={gi+1}: raw.sum={raw[gi].sum():.0f}  eff.sum={eff[gi].sum():.0f}  '
          f'max mult raw={raw[gi].max():.0f} -> eff={eff[gi].max():.0f}')
print(f'\nTOTAL raw={tot_raw:.0f}  eff={tot_eff:.0f}  '
      f'reduction={100*(1-tot_eff/tot_raw):.1f}%')
assert tot_eff < tot_raw, 'kQ/I collapses nothing!'
print('Walk-operator check PASSED — kQ/I strictly lowers the multiplicity to squash.')